# ⚡ StreamVault Cloud Leech Worker (Google Colab Edition)

This notebook downloads massive movies and **web series season packs (>25 GB)** at **50 to 200 MB/s** using Google Cloud data center gigabit fiber, automatically splits episodes >1.95 GB, and uploads them directly into your private Telegram Vault.

### 🚀 Key Advantages:
* **Mobile Hotspot Data Used:** **0 MB** (Google Cloud handles all downloading and Telegram uploading).
* **Massive Web Series Packs:** Downloads entire seasons (30–60 GB), extracts every episode, and uploads them sequentially.
* **Auto-Splitting:** Losslessly splits any episode exceeding 1.95 GB using `ffmpeg -c copy` so Telegram Free Free-Tier never rejects files.
* **Instant Vault & Stream Indexing:** Saves every episode to your MongoDB Atlas collection, instantly streamable via your Oracle VM Stream Server.
* **Two Modes Available:**
  1. **Mode A: Standalone Downloader (On-Demand)** — Paste any magnet link directly into the Colab form and click Run.
  2. **Mode B: MongoDB Queue Worker** — Continuously monitors your bot's `leech_queue`.

### 1️⃣ Install System Tools & High-Speed MTProto Libraries
Run this cell once when starting or reconnecting your Colab session.

In [ ]:
#@title 📦 Install Dependencies
!apt-get update -qq && apt-get install -y -qq aria2 ffmpeg
!pip install -q hydrogram tgcrypto cryptg motor certifi requests python-dotenv nest_asyncio
print("✅ System packages (aria2, ffmpeg) and Python libraries installed successfully!")

### 2️⃣ Configure StreamVault Credentials
You can either:
- **Option 1 (Fastest):** Upload your `.env` file into the Colab file browser on the left (📁 icon) and leave fields as is.
- **Option 2:** Paste your credentials directly into the form fields below.

In [ ]:
#@title ⚙️ Enter Credentials
import os
from dotenv import load_dotenv

# Auto-load .env if uploaded to Colab workspace
if os.path.exists(".env"):
    load_dotenv(".env", override=True)
    print("📄 Detected and loaded .env file from Colab workspace!")

API_ID = 0  #@param {type:"integer"}
API_HASH = ""  #@param {type:"string"}
BOT_TOKEN = ""  #@param {type:"string"}
LOG_CHANNEL_ID = 0  #@param {type:"integer"}
MONGO_URI = ""  #@param {type:"string"}
STREAM_URL = "http://140.245.193.224:7860"  #@param {type:"string"}

# Fallback to environment variables if parameters were left empty
API_ID = API_ID or int(os.getenv("TELEGRAM_API_ID") or 0)
API_HASH = API_HASH or os.getenv("TELEGRAM_API_HASH") or ""
BOT_TOKEN = BOT_TOKEN or os.getenv("TELEGRAM_BOT_TOKEN") or ""
raw_log = str(LOG_CHANNEL_ID or os.getenv("LOG_CHANNEL_ID") or "0").strip()
if raw_log and not raw_log.startswith("-") and raw_log != "0":
    raw_log = f"-{raw_log}"
LOG_CHANNEL_ID = int(raw_log)
MONGO_URI = MONGO_URI or os.getenv("MONGO_URI") or ""
STREAM_URL = STREAM_URL or os.getenv("STREAM_URL") or "http://140.245.193.224:7860"

if all([API_ID, API_HASH, BOT_TOKEN, LOG_CHANNEL_ID, MONGO_URI]):
    print("✅ StreamVault credentials successfully loaded and ready!")
else:
    print("⚠️ Please fill in all required fields or upload your .env file.")

### 3️⃣ Common Download, Split & Upload Engine
Run this cell once to initialize the core streaming and leeching engine.

In [ ]:
import os, sys, re, math, time, shutil, asyncio, subprocess, certifi, nest_asyncio
nest_asyncio.apply()
from motor.motor_asyncio import AsyncIOMotorClient
from hydrogram import Client

TELEGRAM_MAX_BYTES = 1950 * 1024 * 1024  # 1.95 GB Free Limit
VIDEO_EXTENSIONS = (".mp4", ".mkv", ".avi", ".webm", ".ts", ".m4v", ".mov")

def format_size(bytes_size):
    for u in ["B","KB","MB","GB","TB"]:
        if bytes_size < 1024.0: return f"{bytes_size:.2f} {u}"
        bytes_size /= 1024.0
    return f"{bytes_size:.2f} PB"

def natural_sort_key(s):
    return [int(text) if text.isdigit() else text.lower() for text in re.split(r'(\d+)', s)]

def find_all_video_files(download_dir):
    """Finds all video files (episodes), sorting naturally so S01E01 comes before S01E02."""
    videos = []
    for root, _, files in os.walk(download_dir):
        for f in files:
            if f.lower().endswith(VIDEO_EXTENSIONS):
                p = os.path.join(root, f)
                if os.path.getsize(p) > 20 * 1024 * 1024:  # filter samples <20MB
                    videos.append(p)
    videos.sort(key=lambda x: natural_sort_key(os.path.basename(x)))
    return videos

def split_video(path):
    """Losslessly splits video into <1.95 GB chunks if needed."""
    s = os.path.getsize(path)
    if s <= TELEGRAM_MAX_BYTES: return [path]
    try:
        probe_cmd = ["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "default=noprint_wrappers=1:nokey=1", path]
        d = float(subprocess.check_output(probe_cmd).decode().strip())
    except Exception as e:
        print(f"⚠️ ffprobe duration error ({e}), skipping split.")
        return [path]
    parts = math.ceil(s / TELEGRAM_MAX_BYTES)
    if parts < 2 or d <= 0: return [path]
    seg = d / parts
    b, ext = os.path.splitext(path)
    out = f"{b}.part%02d{ext}"
    print(f"✂️ Splitting '{os.path.basename(path)}' ({format_size(s)}) into {parts} parts...")
    split_cmd = ["ffmpeg", "-y", "-i", path, "-c", "copy", "-map", "0", "-f", "segment", "-segment_time", str(seg), "-reset_timestamps", "1", out]
    subprocess.run(split_cmd, check=True)
    dirname = os.path.dirname(path) or "."
    prefix = os.path.basename(b) + ".part"
    res = [os.path.join(dirname, f) for f in sorted(os.listdir(dirname)) if f.startswith(prefix) and f.endswith(ext)]
    return res or [path]

def run_aria2c_torrent(magnet, out_dir="temp_leech"):
    """Downloads torrent at full Google gigabit speed with DHT and peer exchange."""
    os.makedirs(out_dir, exist_ok=True)
    cmd = [
        "aria2c",
        "--seed-time=0",
        "--bt-stop-timeout=600",
        "--enable-dht=true",
        "--bt-enable-lpd=true",
        "--max-connection-per-server=16",
        "-s16", "-x16", "-k1M",
        "--dir", out_dir,
        magnet
    ]
    print(f"🚀 Downloading with aria2c into {out_dir}...")
    subprocess.run(cmd, check=True)
    return find_all_video_files(out_dir)

async def upload_and_index_series(bot, db, videos, title, user_id=None, out_dir=None):
    total_vids = len(videos)
    print(f"🎬 Found {total_vids} video episode(s) to upload into Vault!")
    
    for v_idx, vfile in enumerate(videos, 1):
        base_name = os.path.basename(vfile)
        v_size = os.path.getsize(vfile)
        ep_label = f"[{v_idx}/{total_vids}] " if total_vids > 1 else ""
        print(f"\n{ep_label}Processing: '{base_name}' ({format_size(v_size)})")
        
        parts = split_video(vfile)
        for p_idx, part in enumerate(parts, 1):
            p_name = os.path.basename(part)
            p_size = os.path.getsize(part)
            part_lbl = f" (Part {p_idx}/{len(parts)})" if len(parts) > 1 else ""
            
            caption = f"🎬 {p_name}{part_lbl}"
            print(f"  📤 Uploading '{p_name}' ({format_size(p_size)}) to Telegram Vault...")
            
            msg = await bot.send_document(LOG_CHANNEL_ID, part, caption=caption)
            media = msg.video or msg.document
            
            # Index in MongoDB for immediate streaming
            doc = {
                "file_name": p_name,
                "file_size": p_size,
                "file_id": media.file_id,
                "unique_id": media.file_unique_id,
                "source_chat_id": LOG_CHANNEL_ID,
                "mime_type": "video/mp4",
                "caption": caption
            }
            await db["files"].update_one({"unique_id": media.file_unique_id}, {"$set": doc}, upsert=True)
            
            if user_id:
                stream_link = f"{STREAM_URL.rstrip('/')}/stream/{media.file_unique_id}" if STREAM_URL else "In Vault"
                await bot.send_cached_media(
                    user_id,
                    media.file_id,
                    caption=f"✅ **Ready to Stream:** `{p_name}`{part_lbl}\n📦 `{format_size(p_size)}`\n🔗 `{stream_link}`"
                )
                
    if out_dir and os.path.exists(out_dir):
        shutil.rmtree(out_dir, ignore_errors=True)
    print(f"\n🎉 All {total_vids} episode(s) uploaded and indexed into StreamVault!")

### 4️⃣ Mode A: Standalone On-Demand Leech (Paste Magnet & Run)
Use this cell to download any massive web series season pack (>25 GB) or movie on-demand.

In [ ]:
#@title 📥 Download Web Series / Movie On-Demand
MAGNET_LINK = ""  #@param {type:"string"}
TITLE = "Web Series Pack"  #@param {type:"string"}
DELIVER_TO_USER_ID = 1882765881  #@param {type:"integer"}

async def run_standalone_leech():
    if not MAGNET_LINK.strip():
        print("❌ Please paste a valid magnet link above!")
        return
    bot = Client("colab_standalone", api_id=API_ID, api_hash=API_HASH, bot_token=BOT_TOKEN, in_memory=True)
    mongo = AsyncIOMotorClient(MONGO_URI, tlsCAFile=certifi.where())
    db = mongo["stream_vault"]
    
    await bot.start()
    try:
        if DELIVER_TO_USER_ID:
            await bot.send_message(DELIVER_TO_USER_ID, f"⚡ **Google Colab Leech Started:** `{TITLE}`\nDownloading at Google gigabit speeds (0 MB hotspot data)...")
        out_dir = f"downloads_{int(time.time())}"
        videos = run_aria2c_torrent(MAGNET_LINK, out_dir)
        if not videos:
            print("❌ No video files found in torrent.")
            return
        await upload_and_index_series(bot, db, videos, TITLE, user_id=DELIVER_TO_USER_ID, out_dir=out_dir)
        if DELIVER_TO_USER_ID:
            await bot.send_message(DELIVER_TO_USER_ID, f"🎉 **Complete!** All episodes of `{TITLE}` are in your Vault and ready to stream!")
    finally:
        await bot.stop()

await run_standalone_leech()

### 5️⃣ Mode B: Continuous Queue Worker
Run this cell if you want Colab to listen to your Telegram Bot's background `leech_queue` and process downloads automatically.

In [ ]:
async def run_queue_worker():
    bot = Client("colab_queue_worker", api_id=API_ID, api_hash=API_HASH, bot_token=BOT_TOKEN, in_memory=True)
    mongo = AsyncIOMotorClient(MONGO_URI, tlsCAFile=certifi.where())
    db = mongo["stream_vault"]
    
    await bot.start()
    print("⚡ Colab Queue Worker Online! Listening to MongoDB 'leech_queue'...")
    try:
        while True:
            task = await db["leech_queue"].find_one_and_update(
                {"status": "queued"},
                {"$set": {"status": "processing"}}
            )
            if not task:
                await asyncio.sleep(5)
                continue
            
            title = task.get("title", "Unknown Release")
            user_id = task.get("user_id")
            magnet = task.get("magnet")
            print(f"[*] Processing queued task: {title}")
            
            try:
                if user_id:
                    await bot.send_message(user_id, f"⚡ **Colab Leeching:** `{title}`\nDownloading at Google gigabit speed (0 MB hotspot data)...")
                out_dir = f"temp_{task['_id']}"
                videos = run_aria2c_torrent(magnet, out_dir)
                await upload_and_index_series(bot, db, videos, title, user_id=user_id, out_dir=out_dir)
                await db["leech_queue"].update_one({"_id": task["_id"]}, {"$set": {"status": "completed"}})
                print(f"[+] Task completed: {title}")
            except Exception as e:
                print(f"[!] Task failed: {e}")
                await db["leech_queue"].update_one({"_id": task["_id"]}, {"$set": {"status": "failed", "error": str(e)}})
                if user_id:
                    await bot.send_message(user_id, f"❌ **Leech Failed for {title}:** `{e}`")
    finally:
        await bot.stop()

await run_queue_worker()